# 🚀 📱 Выпускной проект 2: Продуктовая аналитика B2B SaaS и статистический A/B аудит

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Enlyl/data-analyst-2027/blob/main/course/module_15_portfolio_project_2/project_2_saas_product_analytics.ipynb)

## 📌 Бизнес-контекст и методология
### 📌 Паспорт проекта
* **Продукт**: Облачная платформа аналитики подписок (SaaS).
* **Бизнес-задача**:
  1. Выявить скрытые триггеры оттока (Churn Analysis) на базе поведения пользователей (`saas_users.csv`).
  2. Провести сквозной статистический аудит A/B-эксперимента новой формы онбординга (`ab_test_checkout.csv`): проверка SRM, расчет Z-теста и бутстрэпа, экономическая оценка раскатки.

In [1]:
# 🚀 Универсальная инициализация окружения и данных
import os
import urllib.request
import sqlite3
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def ensure_dataset(filename, subfolder=""):
    """Автоматически находит датасет локально или скачивает с GitHub репозитория курса."""
    if os.path.exists(filename):
        return filename
    candidates = [
        os.path.join("course", "datasets", subfolder, filename),
        os.path.join("course", "datasets", filename),
        os.path.join("..", "datasets", subfolder, filename),
        os.path.join("..", "datasets", filename),
        os.path.join("..", "..", "course", "datasets", subfolder, filename),
        os.path.join("datasets", subfolder, filename),
        os.path.join("datasets", filename),
    ]
    for c in candidates:
        if os.path.exists(c):
            return c
    raw_base = "https://raw.githubusercontent.com/Enlyl/data-analyst-2027/main/course/datasets"
    url = f"{raw_base}/{subfolder}/{filename}" if subfolder else f"{raw_base}/{filename}"
    print(f"📥 Загрузка {filename} из репозитория курса...")
    urllib.request.urlretrieve(url, filename)
    print(f"✅ Датасет {filename} успешно готов!")
    return filename

# Загрузка базы данных e-commerce и csv файлов
db_path = ensure_dataset("ecommerce.db", subfolder="")
saas_csv = ensure_dataset("saas_users.csv", subfolder="raw")
ab_csv = ensure_dataset("ab_test_checkout.csv", subfolder="raw")
reviews_csv = ensure_dataset("customer_reviews.csv", subfolder="raw")
print("🟢 Окружение и все корпоративные датасеты готовы к анализу.")


🟢 Окружение и все корпоративные датасеты готовы к анализу.


## 🔍 Разбор механики и демонстрация на реальных данных

In [2]:
# ЭТАП 1: Аудит факторов оттока (Churn Drivers)
import pandas as pd
import numpy as np
from scipy import stats

df_saas = pd.read_csv(saas_csv)

# Анализ зависимости оттока от использования фич (Feature Adoption)
feature_usage_churn = df_saas.groupby('is_churned').agg(
    avg_nps=('nps_score', 'mean'),
    avg_seats=('active_seats', 'mean'),
    avg_feature_a=('feature_a_usage', 'mean'),
    avg_feature_b=('feature_b_usage', 'mean'),
    avg_mrr=('mrr', 'mean')
).reset_index()

print("📊 Сравнение характеристик активных (0) и оттекших (1) клиентов:")
print(feature_usage_churn)


📊 Сравнение характеристик активных (0) и оттекших (1) клиентов:
   is_churned   avg_nps  avg_seats  avg_feature_a  avg_feature_b    avg_mrr
0           0  8.342548  13.537260     250.617788      40.617788  84.318510
1           1  8.357143  12.958333     253.925595      42.901786  52.690476


### 🎯 ЭТАП 2: Статистический анализ A/B теста онбординга
1. Проверьте A/B тест на Sample Ratio Mismatch (SRM) через `stats.chisquare`.
2. Рассчитайте конверсию групп A (Control) и B (Variant).
3. Проведите двусторонний Z-тест для разницы конверсий и сделайте вывод о значимости.

In [3]:
### ВАШ КОД ЗДЕСЬ
df_ab = pd.read_csv(ab_csv)

# 1. SRM Check
counts = df_ab['test_group'].value_counts()
stat_srm, p_srm = stats.chisquare(counts.values, [len(df_ab)/2, len(df_ab)/2])
print(f"SRM Chi2: {stat_srm:.3f}, p-value: {p_srm:.4f}")
assert p_srm > 0.01, "SRM не обнаружен, группы разделены честно"

# 2. Conversion Analysis
cr_a = df_ab[df_ab['test_group'] == 'A_Control']['converted'].mean()
cr_b = df_ab[df_ab['test_group'] == 'B_Variant']['converted'].mean()
n_a = len(df_ab[df_ab['test_group'] == 'A_Control'])
n_b = len(df_ab[df_ab['test_group'] == 'B_Variant'])

uplift_rel = (cr_b - cr_a) / cr_a
pooled_p = (cr_a * n_a + cr_b * n_b) / (n_a + n_b)
se = np.sqrt(pooled_p * (1 - pooled_p) * (1/n_a + 1/n_b))
z_score = (cr_b - cr_a) / se
p_value = 2 * (1 - stats.norm.cdf(abs(z_score)))

print(f"CR A: {cr_a:.2%}, CR B: {cr_b:.2%}, Относительный прирост: +{uplift_rel:.1%}")
print(f"Z-score: {z_score:.3f}, p-value: {p_value:.4e}")

is_significant = p_value < 0.05
print(f"Статистическая значимость на уровне alpha=0.05: {is_significant}")

assert not is_significant, "На текущей выборке различия не достигли порога alpha=0.05"
assert uplift_rel > 0, "Наблюдаемый точечный uplift положителен, но статистически не подтвержден"
print("✅ Тест успешно пройден: аналитик избежал ложноположительной ошибки (False Discovery)!")


SRM Chi2: 0.000, p-value: 1.0000
CR A: 8.50%, CR B: 9.26%, Относительный прирост: +8.9%
Z-score: 1.336, p-value: 1.8159e-01
Статистическая значимость на уровне alpha=0.05: False
✅ Тест успешно пройден: аналитик избежал ложноположительной ошибки (False Discovery)!


### 💼 ЭТАП 3: C-Level вердикт по эксперименту
* Наблюдаемый прирост конверсии составляет +8.9%, однако $p = 0.18 > 0.05$.
* Риск ошибки I рода (ложная тревога) превышает допустимый порог.
* **Управленческое решение**: Раскатку на 100% заморозить до сбора расчетного Sample Size либо признать эффект статистическим шумом, сэкономив ресурсы разработки на поддержку сырой функциональности.